# Import data

In [ ]:
import cv2
import matplotlib.pyplot as plt

def imshow(filename):
  image = cv2.imread(filename)

  plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
  plt.axis('off')
  plt.show()
  print(image.shape)

In [ ]:
path = 'D:/programing/Final Project/Alzheimer/dataset/Dataset/'

In [ ]:
imshow(path+'Mild_Demented/mild.jpg')

# Image Preprocessing

In [ ]:
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, WeightedRandomSampler
from PIL import Image
import numpy as np
import random
random.seed(0)
torch.manual_seed(0)
np.random.seed(0)
torchvision.disable_beta_transforms_warning()

## CustomDataset to import dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
import copy
import random

class CustomDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = os.listdir(root_dir)
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}
        self.images = self._get_images()
        self.class_count = self._get_class_count()
        self.images_splited_each_class = self._split_img_each_class()
        self.random_dataset = self.random_data()

    def _get_images(self):
        images = []
        for cls in self.classes:
            class_path = os.path.join(self.root_dir, cls)
            class_idx = self.class_to_idx[cls]
            for img_name in os.listdir(class_path):
                img_path = os.path.join(class_path, img_name)
                images.append((img_path, class_idx))
        return images
    
    def _get_class_count(self):
        class_count = {}
        for cls_name in self.class_to_idx.values():
            class_count[cls_name] = 0
        for key,value in self.images:
           class_count[value] += 1 
        return class_count
    
    def __len__(self):
        return len(self.images)
       
    def _get_labels(self):
        return [i[1] for i in self.images]
    
    def _split_img_each_class(self):
        data_class_split = {}
        for i in self.class_to_idx.values():
            data_class_split[i] = []
        for i in self.class_to_idx.values():
            for j in range(len(self.images)):
                if self.images[j][-1] == i:
                    data_class_split[i].append(self.images[j])
        return data_class_split
    
    def random_data(self):
        images_splited_each_class = copy.deepcopy(self.images_splited_each_class)
        random_list_idx = ['unknown'] + [i for i in range(len(self.classes))]
        allrandom_data = []

        for i in range(len(self.images)):
            random_data = []
            for class_idx in random_list_idx:
                if class_idx == 'unknown':
                    #unknown class
                    randclass = torch.randint(low=0, high=len(self.classes), size=(1,)).item()
                    random_img_unk = random.choice(images_splited_each_class[randclass])
                    random_data.append(random_img_unk)
                else:
                    #other class
                    random_img = random.choice(images_splited_each_class[class_idx])
                    random_data.append(random_img)
            allrandom_data.append(random_data)
        return allrandom_data

    def __getitem__(self, idx):
        data = self.random_dataset[idx]
        allrandom_img = []
        allrandom_label = []

        for img_path, label in data:
            img = Image.open(img_path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            allrandom_img.append(img)
            allrandom_label.append(label)

        return allrandom_img, allrandom_label

In [ ]:
sample_data = CustomDataset(root_dir='D:/programing/Final Project/Alzheimer/dataset/Dataset')

In [ ]:
import matplotlib.image as mpimg
import copy

sample_data_dict = copy.deepcopy(sample_data.images_splited_each_class)

# create 4x1 subplots
fig, axs = plt.subplots(nrows=4, ncols=1, constrained_layout=True, figsize=(10, 10))
fig.suptitle('Sample 20 data (5 images for each class)')

# clear subplots
for ax in axs:
    ax.remove()

# add subfigure per subplot
gridspec = axs[0].get_subplotspec().get_gridspec()
subfigs = [fig.add_subfigure(gs) for gs in gridspec]

for row, subfig in enumerate(zip(subfigs,sample_data.classes)):
    subfig[0].suptitle(subfig[-1])

    # create 1x5 subplots per subfig
    axs = subfig[0].subplots(nrows=1, ncols=5)

    for class_label, images in sample_data_dict.items():
        for i in range(5):
            img = mpimg.imread(images[i][0])
            axs[i].imshow(img,cmap='gray')
            axs[i].axis('off')
        break
 
    del sample_data_dict[class_label]

plt.show()

## Split white and gray matter

In [ ]:
class split_white_and_gray():
    def __init__(self,threshold=120) -> None:
        self.threshold = threshold

    def __call__(self,tensor):
        tensor = (tensor*255).to(torch.int64)

        # Apply thresholding
        gray_matter = torch.where(tensor >= self.threshold,tensor,0)
        gray_matter = (gray_matter/255).to(torch.float64)
        white_matter = torch.where(tensor < self.threshold,tensor,0)
        white_matter = (white_matter/255).to(torch.float64)
        tensor = (tensor/255).to(torch.float64)

        return torch.cat((gray_matter, white_matter,tensor), dim=0)

# Prepare Dataset 

## Data transforms

In [ ]:
transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    split_white_and_gray(120),
])

In [ ]:
Dataset = CustomDataset(root_dir='D:/programing/Final Project/Alzheimer/dataset/Dataset', transform=transform)

In [ ]:
plt.imshow(transforms.ToPILImage()(Dataset[0][0][0][0]),cmap='gray')
plt.show()

In [ ]:
plt.imshow(transforms.ToPILImage()(Dataset[0][0][0][1]),cmap='gray')
plt.show()

## Split dataset 

In [ ]:
from torch.utils.data import random_split

# Assuming you have a dataset named 'dataset' and it contains your data
dataset_size = len(Dataset)
train_size = int(0.4 * dataset_size)  # 70% for training
val_size = int(0.4 * dataset_size)  # 15% for validation
test_size = dataset_size - train_size - val_size  # Remaining 15% for testing

train_dataset, val_test_dataset = random_split(Dataset, [train_size, val_size + test_size])
val_dataset, test_dataset = random_split(val_test_dataset, [val_size, test_size])

## Apply DataLoader

In [ ]:
batch_size=64

train_dataloader = DataLoader(train_dataset, batch_size=batch_size)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)


In [ ]:
class_counts = {label: 0 for label in range(len(Dataset.classes))}

# Iterate through the DataLoader
for _, labels in train_dataloader:
    for label in labels[0]:
        class_counts[label.item()] += 1

# Print the counts
for label, count in class_counts.items():
    print(f"Class {label}: {count} items")

In [ ]:
# Create a figure with 1 row and 2 columns
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))  # Adjust the figsize as needed

# Add the first subplot
ax1.bar(Dataset.classes, Dataset.class_count.values())
ax1.set_xticks(range(len(Dataset.classes)))
ax1.set_xticklabels(Dataset.classes, rotation=45)
ax1.set_ylim(top=6000)
ax1.set_title("Before Over sampling")

# Add the second subplot
ax2.bar(Dataset.classes, class_counts.values())
ax2.set_xticks(range(len(Dataset.classes)))
ax2.set_xticklabels(Dataset.classes, rotation=45)
ax2.set_ylim(top=6000)
ax2.set_title("After Over sampling")

plt.show()


## visualization of each class

In [ ]:
images, labels = next(iter(train_dataloader))

img_index = []
for label in labels[0].unique():
    img_index.append(list(labels[0]).index(label))

image_each_classes = []
for idx in img_index:
    image_each_classes.append(images[0][idx])

# Plot one image from each class
fig, axs = plt.subplots(1, len(Dataset.classes), figsize=(15, 3))
fig.suptitle("White matter from Each Class", fontsize=16)

for class_num, i in enumerate(zip(Dataset.classes,image_each_classes)):
    axs[class_num].imshow(transforms.ToPILImage()(i[-1][0]),cmap='gray')
    axs[class_num].set_title(f"Class {i[0]}")
    axs[class_num].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Plot one image from each class
fig, axs = plt.subplots(1, len(Dataset.classes), figsize=(15, 3))
fig.suptitle("Gray matter from Each Class", fontsize=16)

for class_num, i in enumerate(zip(Dataset.classes,image_each_classes)):
    axs[class_num].imshow(transforms.ToPILImage()(i[-1][1]),cmap='gray')
    axs[class_num].set_title(f"Class {i[0]}")
    axs[class_num].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Plot one image from each class
fig, axs = plt.subplots(1, len(Dataset.classes), figsize=(15, 3))
fig.suptitle("Original image from Each Class", fontsize=16)

for class_num, i in enumerate(zip(Dataset.classes,image_each_classes)):
    axs[class_num].imshow(transforms.ToPILImage()(i[-1][2]),cmap='gray')
    axs[class_num].set_title(f"Class {i[0]}")
    axs[class_num].axis('off')

plt.tight_layout()
plt.show()

# Backbone FeatureExtractor

class names : https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json,
https://github.com/anishathalye/imagenet-simple-labels/blob/master/imagenet-simple-labels.json

In [ ]:
import requests

# Fetch the ImageNet class labels
LABELS_URL = 'https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json'
response = requests.get(LABELS_URL)
labels = response.json()

labels[:10]

## Resnet18 transfer model

### Cos similarity

In [ ]:
import torch
import torch.nn.functional as F

# Assuming input_batch and ground_truth_batch are your tensors
input_batch = torch.randn(64, 3, 150, 150)
ground_truth_batch = torch.randn(4, 3, 150, 150)

# Reshape input_batch for cosine similarity calculation
input_batch_reshaped = input_batch.view(64, -1)  # 64 images in batch, flattened

# Reshape ground_truth_batch for cosine similarity calculation
ground_truth_batch_reshaped = ground_truth_batch.view(4, -1)  # 4 representative images per class, flattened

# Calculate cosine similarity
cosine_sim = F.cosine_similarity(input_batch_reshaped.unsqueeze(1), ground_truth_batch_reshaped.unsqueeze(0), dim=2)

# Result: cosine_sim is a tensor of size (64, 4), containing cosine similarity values for each pair of images

### Siamese Network

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import collections
import torch.optim as optim
from sklearn.metrics import accuracy_score

class SiameseNetwork_Resnet18(nn.Module):
    def __init__(self):
        super(SiameseNetwork_Resnet18, self).__init__()
        self.resnet18 = models.resnet18()
        self.resnet18_removelast = nn.Sequential(*list(self.resnet18.children())[:-1])
        self.batchnorm = nn.BatchNorm1d(128)
        self.classification_head = nn.Sequential(
                                                    nn.Linear(512, 256),  # Linear layer
                                                    nn.ReLU(inplace=True),
                                                    nn.Linear(256, 128),
                                                    nn.ReLU(inplace=True),
                                                )
        self.resnet18_backbone = nn.Sequential(
                                                    collections.OrderedDict(
                                                        [
                                                            ("resnet18", self.resnet18_removelast),
                                                            ("Flatten", nn.Flatten()),  # Flatten the output
                                                            ("Head", self.classification_head),
                                                        ]
                                                    )
                                                )


    def forward_once(self, x):
        return self.resnet18_backbone(x)
    
    def zero_allmin(self,x):
        y = x.clone()
        y[x == x.min(dim=1, keepdims=True).values] = 0.0
        return y

    def forward(self, unknown_input, input0, input1, input2, input3):
        unknown_output = self.batchnorm(self.forward_once(unknown_input))
        output0 = self.batchnorm(self.forward_once(input0))
        output1 = self.batchnorm(self.forward_once(input1))
        output2 = self.batchnorm(self.forward_once(input2))
        output3 = self.batchnorm(self.forward_once(input3))

        # Calculate cosine similarity
        similarity_0 = F.cosine_similarity(unknown_output, output0)
        similarity_1 = F.cosine_similarity(unknown_output, output1)
        similarity_2 = F.cosine_similarity(unknown_output, output2)
        similarity_3 = F.cosine_similarity(unknown_output, output3)
        del unknown_output, output0, output1, output2, output3
        
        # Stack the linear outputs into a vector
        similarities = torch.stack([similarity_0, similarity_1, similarity_2, similarity_3], dim=1)
        del similarity_0, similarity_1, similarity_2, similarity_3
        
        # Normalize the tensor
        min_values, _ = torch.min(similarities, dim=1, keepdim=True)
        max_values, _ = torch.max(similarities, dim=1, keepdim=True)
        normalized_tensor = (similarities - min_values) / (max_values - min_values)

        similarities = self.zero_allmin(normalized_tensor)
        probability = torch.softmax(similarities, dim=1)

        return probability


In [ ]:
# Move the model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
snr = SiameseNetwork_Resnet18().float().to(device)

In [ ]:
a = next(iter(train_dataloader))[0]
i1 = a[0][0].to(torch.float).unsqueeze(0)
i2 = a[1][0].to(torch.float).unsqueeze(0)
i3 = a[2][0].to(torch.float).unsqueeze(0)
i4 = a[2][0].to(torch.float).unsqueeze(0)
i5 = a[4][0].to(torch.float).unsqueeze(0)


In [ ]:
k = torch.randn((2,3,20,20))

In [ ]:
snr.eval()
o = snr(k,k,k,k,torch.randn((2,3,20,20)))

In [ ]:
o

### Layers and Params

In [ ]:
# Print parameters for each layer
for name, param in snr.named_parameters():
    print(f"Layer: {name}, Size: {param.size()}, Parameters: {param.numel()}")

In [ ]:
sum(p.numel() for p in snr.parameters())

### Draw graph

In [ ]:
# from torchviz import make_dot

# # Generate a visual representation of the model's computational graph
# y = snr(i1,i2,i3,i4,i5)
# dot = make_dot(y, params=dict(snr.named_parameters()))

# # Save the generated graph as an image
# dot.render("your_model_graph")

### training

In [ ]:
from torch.optim import lr_scheduler
import tqdm.notebook as tq

# training
# Loop for training
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(snr.parameters(), lr=0.001)
scheduler = lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

num_epochs = 2

for epoch in tq.tqdm(range(num_epochs), desc="Epoch num", position=0, leave=True):
    # Train set
    running_loss = 0.0
    all_labels = []
    all_predictions = []

    # Val set
    val_running_loss = 0.0
    val_all_labels = []
    val_all_predictions = []

    # Use tqdm for progress visualization
    total_batches = min(len(train_dataloader), len(val_dataloader))  # Ensure the total_batches is not larger than the smallest of the two
    for i, data in tq.tqdm(enumerate(zip(train_dataloader, val_dataloader)),desc="Batch num", position=0, leave=True, total=total_batches):
        snr.train()#change mode to training

        train_set, val_set = data
        inputs, labels = train_set

        optimizer.zero_grad()

        # Move inputs and labels to device
        input_0 = inputs[0].to(torch.float32).to(device)
        input_1 = inputs[1].to(torch.float32).to(device)
        input_2 = inputs[2].to(torch.float32).to(device)
        input_3 = inputs[3].to(torch.float32).to(device)
        input_4 = inputs[4].to(torch.float32).to(device)

        outputs = snr(input_0, input_1, input_2, input_3, input_4)
        loss = criterion(outputs, labels[0])
        loss.backward()
        optimizer.step()

        # Track predictions for accuracy calculation
        _, predictions = torch.max(outputs, 1)
        all_labels.extend(labels[0].to(device).numpy())
        all_predictions.extend(predictions.cpu().numpy())

        # Accumulate the loss
        running_loss += loss.item()
#################################################################################################        
        # loss and acc on validation set
        snr.eval() #change mode to evaluation
        with torch.no_grad():
            val_inputs, val_labels = val_set
            val_input_0 = val_inputs[0].to(torch.float32).to(device)
            val_input_1 = val_inputs[1].to(torch.float32).to(device)
            val_input_2 = val_inputs[2].to(torch.float32).to(device)
            val_input_3 = val_inputs[3].to(torch.float32).to(device)
            val_input_4 = val_inputs[4].to(torch.float32).to(device)

            val_outputs = snr(val_input_0, val_input_1, val_input_2, val_input_3, val_input_4)
            val_loss = criterion(val_outputs, val_labels[0])

            _, val_predictions = torch.max(val_outputs, 1)
            val_all_labels.extend(val_labels[0].to(device).numpy())
            val_all_predictions.extend(val_predictions.cpu().numpy())

            val_running_loss += val_loss.item()
#################################################################################################
    scheduler.step()
    
    # Calculate accuracy on training set
    accuracy = accuracy_score(all_labels, all_predictions)

    # Calculate accuracy on validation set
    val_accuracy = accuracy_score(val_all_labels, val_all_predictions)
    
    # Print statistics for the epoch
    print("-------------------------------------------------------------------")
    print(f"Epoch {epoch + 1}/{num_epochs}")
    print(f"Training Loss   : {running_loss / len(train_dataloader):.3f}  |  Training Accuracy   : {accuracy:.3f}")
    print(f"Validation Loss : {val_running_loss / len(val_dataloader):.3f}  |  Validation Accuracy : {val_accuracy:.3f}")
    print("-------------------------------------------------------------------")
    print()

In [ ]:
# Move model to device outside the loop
snr.to(device)

for epoch in tq.tqdm(range(num_epochs), desc="Epoch num", position=0, leave=True):
    # Switch between training and evaluation modes
    snr.train()
    running_loss = 0.0
    all_labels = []
    all_predictions = []

    snr.eval()
    val_running_loss = 0.0
    val_all_labels = []
    val_all_predictions = []

    # Use DataLoader directly for batching
    for train_set, val_set in zip(train_dataloader, val_dataloader):
        inputs, labels = train_set
        val_inputs, val_labels = val_set

        optimizer.zero_grad()

        # Move inputs and labels to device
        inputs = [inp.to(torch.float32).to(device) for inp in inputs]
        outputs = snr(*inputs)
        loss = criterion(outputs, labels[0])
        
        if snr.training:
            loss.backward()
            optimizer.step()

        # Track predictions for accuracy calculation
        _, predictions = torch.max(outputs, 1)
        all_labels.extend(labels[0].to(device).numpy())
        all_predictions.extend(predictions.cpu().numpy())

        # Accumulate the loss
        running_loss += loss.item()

        # Validation set
        with torch.no_grad():
            val_inputs = [inp.to(torch.float32).to(device) for inp in val_inputs]
            val_outputs = snr(*val_inputs)
            val_loss = criterion(val_outputs, val_labels[0])

            _, val_predictions = torch.max(val_outputs, 1)
            val_all_labels.extend(val_labels[0].to(device).numpy())
            val_all_predictions.extend(val_predictions.cpu().numpy())

            val_running_loss += val_loss.item()

    scheduler.step()

    # Calculate accuracy on training set
    accuracy = accuracy_score(all_labels, all_predictions)

    # Calculate accuracy on validation set
    val_accuracy = accuracy_score(val_all_labels, val_all_predictions)

    # Print statistics for the epoch
    print("-------------------------------------------------------------------")
    print(f"Epoch {epoch + 1}/{num_epochs}")
    print(f"Training Loss   : {running_loss / len(train_dataloader):.3f}  |  Training Accuracy   : {accuracy:.3f}")
    print(f"Validation Loss : {val_running_loss / len(val_dataloader):.3f}  |  Validation Accuracy : {val_accuracy:.3f}")
    print("-------------------------------------------------------------------")
    print()


In [ ]:
import torch
import psutil

# After training
current_process = psutil.Process()
memory_info = current_process.memory_info()

print(f"Memory used during training: {memory_info.rss / 1024**2} MB")


### Save model

In [ ]:
#torch.save(resnet18_backbone, "resnet18_backbone.pth")

In [ ]:
#loaded_model = torch.load("resnet18_backbone.pth")

## CONVNEXT_BASE

In [ ]:
# Load pre-trained convnext_base model
convnext_base = models.convnext_base(weights=models.ConvNeXt_Base_Weights.DEFAULT)

modules_except_last = list(convnext_base.children())[:-1]
last_module = list(convnext_base.children())[-1]
modified_last_module = nn.Sequential(*list(last_module.children())[:-1])

convnext_base_removelast = nn.Sequential(*modules_except_last, modified_last_module)

# Add a new classification head for 4 classes
num_classes = 4

# Adjust the size of the new classification head based on the output size
classification_head = nn.Sequential(
    nn.Linear(1024, 512),  # Linear layer
    nn.ReLU(inplace=True),
    nn.Linear(512, 256),
    nn.ReLU(inplace=True),
    nn.Linear(256, num_classes),
    nn.Softmax(dim=1),  # Softmax activation along dimension 1 (assuming 0-indexed)
)

# Combine the feature extraction backbone and the new classification head
convnextbase_backbone = nn.Sequential(
    collections.OrderedDict(
        [
            ("convnextbase", convnext_base_removelast),
            ("Head", classification_head),
        ]
    )
)

# Move the model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
convnextbase_backbone.to(device)

In [ ]:
sum(p.numel() for p in convnextbase_backbone.parameters())

## FeatureExtractor

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

# Define a custom model with a CNN backbone for feature extraction
class FeatureExtractor(nn.Module):
    def __init__(self, cnn_backbone):
        super(FeatureExtractor, self).__init__()
        # Use the specified CNN backbone
        self.cnn_backbone = cnn_backbone

    def forward(self, x):
        # Forward pass through the CNN backbone to obtain feature maps
        features = self.cnn_backbone(x)
        return features

# Load a pre-trained ResNet18 model
resnet18 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Remove the final fully connected layers for classification
# to get the feature extraction backbone
feature_extraction_backbone = nn.Sequential(*list(resnet18.children())[:-2])

# Create an instance of the FeatureExtractor using ResNet18 backbone
feature_extractor = FeatureExtractor(feature_extraction_backbone)

# Set the model to evaluation mode
feature_extractor.eval()

In [ ]:
# Perform feature extraction
with torch.no_grad():
    extracted_features = feature_extractor(images[0][0].unsqueeze(0).to(torch.float32))

# Print the shape of the extracted features
print(extracted_features.shape)


# torchvision feature extraction

credit : https://github.com/alexander-soare/torchvision_feature_extraction_walkthrough/blob/main/notebook.ipynb?fbclid=IwAR3J_jzbGtoChdH20zGemBxqpeJWqoCttwnamx65Bzov57qWHcil_CpzbTg

youtube : https://www.youtube.com/watch?v=QRQBTkCLpFY&t=1592s

## Def files

In [ ]:
import torch
import numpy as np

def prepare(img, mean, std):
    img = img / 255.
    img = img - np.array(mean)
    img = img / np.array(std)
    return torch.tensor(img).permute(2, 0, 1).unsqueeze(0).float()

In [ ]:
from typing import Union, Tuple

import cv2
import numpy as np


def center_crop(img: np.ndarray, crop_size: Tuple[int, int]) -> np.ndarray:
    """
    Args:
        img - Image to be cropped
        crop_size - Size of crop. Must be smaller than or equal to the image
            size
    """
    assert (img.shape[0] >= crop_size[0]) and (img.shape[1] >= crop_size[1])
    y0 = img.shape[0] // 2 - crop_size[0] // 2
    y1 = y0 + crop_size[0]
    x0 = img.shape[1] // 2 - crop_size[1] // 2
    x1 = x0 + crop_size[1]
    return img.copy()[y0:y1, x0:x1]


def resize_shortest_edge(
        img: np.ndarray, length: int,
        interpolation: Union[int, str]=cv2.INTER_LINEAR) -> np.ndarray:
    """
    Resize image with locked aspect ratio such that its shortest side is `length`
    pixels long.
    `interpolation` specifies the cv2 interpolation type and defaults to
    cv2.INTER_LINERAR It may be specified as 'auto' in which case either
    cv2.INTER_AREA or cv2.INTERCUBIC is used depnding on whether we are
    downsizing or upsizing (respectively)
    """
    f = length/np.min(img.shape[:2])
    if isinstance(interpolation, str):
        assert interpolation == 'auto', \
            "If `interpolation` is a str it can only be 'auto'"
        interpolation = cv2.INTER_AREA if f < 1 else cv2.INTER_CUBIC
    return cv2.resize(img, (0,0), fx=f, fy=f, interpolation=interpolation)

In [ ]:
from typing import Sequence

import matplotlib.pyplot as plt
import numpy as np


def show_images(ls_img, titles=[], imsize=(7, 5), cmap=None, per_row=2,
                keep_ticks=False, font_size=16):
    """makes a figure with enough subplots to show the images of `ls_img`
    """                
    # make sure ls_img is a list
    if not isinstance(ls_img, Sequence):
        ls_img = [ls_img]

    # make sure titles is a list
    if not isinstance(titles, Sequence):
        titles = [titles]

    # make sure titles is same length as ls_img
    if len(titles):
        assert len(titles) == len(
            ls_img), "Please provide as many titles as there are images"
    else:
        titles = [''] * len(ls_img)

    # prepare figure
    num_rows = len(ls_img) // per_row + ((len(ls_img) % per_row) > 0)
    fig, ax = plt.subplots(num_rows, per_row, figsize=(
        imsize[0] * per_row, imsize[1] * num_rows))
    if type(ax) == np.ndarray:
        ax = ax.flatten()
    else:
        ax = np.array([ax])

    # populate figure
    for i, img in enumerate(ls_img):
        this_cmap = cmap
        if this_cmap is None and (len(img.shape) == 2 or img.shape[-1] == 1):
            this_cmap = 'gray'
        ax[i].imshow(img, cmap=this_cmap, vmin=0, vmax=255)
        ax[i].set_title(titles[i], fontdict={'fontsize': font_size})
        if not keep_ticks:
            ax[i].set_xticks([])
            ax[i].set_yticks([])
    plt.tight_layout()
    return fig, ax

## Perform backbone from timm

In [ ]:
import timm

model = timm.create_model('resnest50d',pretrained=True)
model.eval()

In [ ]:
model.get_classifier()

In [ ]:
imgs = []

for file_path in [path+'Mild_Demented/mild.jpg']:
    img = cv2.imread(file_path)[..., ::-1]
    img = resize_shortest_edge(img, 384, interpolation='auto')
    img = center_crop(img, (384, 384))
    imgs.append(img)

show_images(imgs, per_row=1, imsize=(5, 5))

In [ ]:
import torch

inps = [prepare(img, model.default_cfg['mean'], model.default_cfg['std'])
        for img in imgs]

with torch.no_grad():
    out = torch.softmax(model(inps[0]), -1)[0].numpy()

print(out.argmax())

In [ ]:
from torchvision.models.feature_extraction import get_graph_node_names
from pprint import pprint

train_nodes, eval_nodes = get_graph_node_names(model)

assert([t == e for t, e in zip(train_nodes, eval_nodes)])

pprint(train_nodes[0:5])

In [ ]:
from torchvision.models.feature_extraction import create_feature_extractor
import random
import seaborn as sns
import matplotlib.pyplot as plt

return_nodes = ['layer1', 'layer2', 'layer3', 'layer4']

feat_ext = create_feature_extractor(model, return_nodes=return_nodes)

with torch.no_grad():
    out = feat_ext(inps[0])

fig, ax = plt.subplots(4, 5, figsize=(25, 20))

# Pick 4 random feature maps from each layer
for i, layer in enumerate(return_nodes):
    feat_maps = out[layer].numpy().squeeze(0)
    feat_maps = random.sample(list(feat_maps), 4)
    ax[i][0].imshow(imgs[0])
    ax[i][0].set_xticks([])
    ax[i][0].set_yticks([])
    for j, feat_map in enumerate(feat_maps):
        sns.heatmap(feat_map, ax=ax[i][j+1], cbar=False)
        ax[i][j+1].set_xticks([])
        ax[i][j+1].set_yticks([])
        ax[i][j+1].set_title(f'{layer}: ({feat_map.shape[0]} x {feat_map.shape[1]})')